# Correlogram


In [ ]:
%load_ext watermark
%watermark -a 'eli knaap'
%load_ext autoreload
%autoreload 2

import geopandas as gpd
from libpysal import examples

from esda import correlogram

In [ ]:
sac = gpd.read_file(examples.load_example("Sacramento1").get_path("sacramentot2.shp"))

In [ ]:
sac = sac.to_crs(sac.estimate_utm_crs())  # now in meters)

In [ ]:
correlogram?

In [ ]:
from esda import G, Geary, Moran

## Distance Bands


In [ ]:
# Create a liste of distances between 500 and 5000 (meters, here) in increments of 500

distances = [i + 500 for i in range(0, 5000, 500)]

In [ ]:
distances

The correlogram will compute an autocorrelation statistic (Moran's $I$ by default) at each distance threshold. Plotting this statistic against distance reveals how spatial similarity changes over distance (similar in concept to a variogram)

In [ ]:
prof = correlogram(sac.centroid, sac.HH_INC, distances, Moran)

`prof` is a dataframe of autocorrelation statistics indexed by distance. It includes all attributes created by the esda autocorrelation statistic class (e.g. [Moran](https://pysal.org/esda/generated/esda.Moran.html#esda.Moran), [Geary](https://pysal.org/esda/generated/esda.Geary.html#esda.Geary), or [Geits-Ord G](https://pysal.org/esda/generated/esda.G.html#esda.G)). The row index for each statistic is the distance at which it was computed

In [ ]:
prof.head()

Often, it is easiest to visualize the statistic, and the pandas `plot` function will plot a column against the dataframe's index by default

In [ ]:
ax = prof.I.plot()
ax.set_xlabel("distance (m)")
ax.set_ylabel("Moran's I")

Autocorrelation statistics differ in concept, so the shape of the spatial correlogram statistic will vary considerably, based on which statistic is created. For example, we can also plot Geary's $C$

In [ ]:
prof = correlogram(sac.centroid, sac.HH_INC, distances, statistic=Geary)

In [ ]:
ax = prof.C.plot()
ax.set_xlabel("distance (m)")
ax.set_ylabel("Geary's C")

...Or Getis-Ord $G$

In [ ]:
prof = correlogram(sac.centroid, sac.HH_INC, distances, statistic=G)

In [ ]:
ax = prof.G.plot()
ax.set_xlabel("distance (m)")
ax.set_ylabel("Getis-Ord G")

You can also leave the distance thresholds unspecified, and the correlogram will estimate over a reasonable range of distances:

In [ ]:
prof = correlogram(sac.centroid, sac.HH_INC, statistic=G)
ax = prof.G.plot()
ax.set_xlabel("distance (m)")
ax.set_ylabel("Getis-Ord G")

## KNN Distance


It is also possible to consider different concepts of *distance*. For example, rather than stepping through increments of Euclidean distance/length at each interval, we could instead step through increments of nearest-neighbors.

Instead of adding neighbors using sequential distances of 500 meters, here we will step through the 50 nearest neighbors, adding one neighbor at a time

In [ ]:
kdists = list(range(1, 50))

In [ ]:
kcorr = correlogram(sac.centroid, sac.HH_INC, kdists, distance_type="knn")

In [ ]:
ax = kcorr.I.plot()
ax.set_xlabel("number of nearest neighbors")
ax.set_ylabel("Moran's I")

## Nonparametric


Finally it is also possible to fit a non-parametric curve to estimate spatial autocorrelation as a function of distance. This is done using a LOWESS smoother, which fits a locally-weighted polynomial regression to the data based on the following regression: 

$$ z_iz_j = f(d_{ij}) + u_{ij}$$

where $z_i$ and $z_j$ are standardized values of the variable of interest at locations $i$ and $j$, $d_{ij}$ is the distance between locations $i$ and $j$, and $u_{ij}$ is an error term. The function $f$ is estimated using a LOWESS smoother, which fits a locally-weighted polynomial regression to the data.

This can be interpreted as the correlation between observations separated by *approximately* the distance on the horizontal axis.

In [ ]:
nonparametric = correlogram(sac.centroid, sac.HH_INC, statistic="lowess")
ax = nonparametric.lowess.plot()
ax.set_xlabel("Distance (m)")
ax.set_ylabel("Correlation between pairs\napproximately this separation")